In [1]:
import os
import mlflow

os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"

mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")

C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/ced16f5d8bce44b5a9c3e335b0f484b4', creation_time=1784571024655, effective_trace_archival_retention=None, experiment_id='9', last_update_time=1784571024655, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [3]:
# Set or create an experiment
#mlflow.set_experiment("ML Algos with HP Tuning")

# Imports
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import ADASYN
from lightgbm import LGBMClassifier
import mlflow.sklearn
import optuna

In [ ]:
# Step 1: Load and clean data
df = pd.read_csv('dataset.csv').dropna()
df = df.dropna(subset=['category'])

# Step 2: Split FIRST on raw text to avoid data leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Step 3: TF-IDF vectorizer - FIT ONLY on training data
ngram_range = (1, 2)  # Bigram
max_features = 10000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train = vectorizer.fit_transform(X_train_raw)  # Fit ONLY on training data
X_test = vectorizer.transform(X_test_raw)  # Transform test data (don't refit)

# Step 4: Apply ADASYN ONLY to training data to handle class imbalance
adasyn = ADASYN(random_state=42)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train, y_train)

# Test data stays untouched for fair evaluation
X_train = X_train_resampled
y_train = y_train_resampled

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type and experiment info
        mlflow.set_tag("mlflow.runName", f"{model_name}_ADASYN_TFIDF_Bigrams_NoLeakage")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.set_tag("data_handling", "no_leakage_fixed")
        mlflow.set_tag("model_type", "LGBMClassifier")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("vectorizer_type", "TfidfVectorizer")
        mlflow.log_param("ngram_range", str(ngram_range))
        mlflow.log_param("vectorizer_max_features", max_features)
        mlflow.log_param("imbalance_handling", "ADASYN_on_train_only")
        mlflow.log_param("n_estimators", model.n_estimators)
        mlflow.log_param("learning_rate", model.learning_rate)
        mlflow.log_param("max_depth", model.max_depth)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report metrics
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        
        print(f"Run logged successfully for {model_name}")


# Step 5: Optuna objective function for LightGBM
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = LGBMClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


# Step 6: Run Optuna for LightGBM, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42, verbose=-1)

    # Log the best model with MLflow
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test)
    
    print(f"\nBest Optuna parameters: {best_params}")
    print(f"Best trial accuracy: {study.best_value:.4f}")

# Run the experiment for LightGBM
run_optuna_experiment()

[I 2026-07-21 01:23:38,281] A new study created in memory with name: no-name-b88b08a3-340e-4089-abf3-debcddc0b2c0
C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-21 01:24:46,445] Trial 0 finished with value: 0.6063002863766535 and parameters: {'n_estimators': 97, 'learning_rate': 0.021913452387494762, 'max_depth': 3}. Best is trial 0 with value: 0.6063002863766535.
C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-21 01:27:07,010] Trial 1 finished with value: 0.585435701622801 and parameters: {'n_estimators': 115, 'learning_rate': 0.002441328258663933, 'max_depth': 8}. Best is trial 0 with value: 0.6063002863766535.
C:\Users\Dell\AppData\Roaming\

🏃 View run LightGBM_ADASYN_TFIDF_Bigrams_NoLeakage at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/9/runs/88233e0638e649a282bb7960daa1b5e6
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/9


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['collections.OrderedDict', 'lightgbm.basic.Booster', 'lightgbm.sklearn.LGBMClassifier'].

In [5]:
# Step 1: Load and clean data
df = pd.read_csv('dataset.csv').dropna()
df = df.dropna(subset=['category'])

# Step 2: Split FIRST on raw text to avoid data leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Step 3: TF-IDF vectorizer - FIT ONLY on training data
ngram_range = (1, 2)  # Bigram
max_features = 10000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train = vectorizer.fit_transform(X_train_raw)  # Fit ONLY on training data
X_test = vectorizer.transform(X_test_raw)  # Transform test data (don't refit)

# Step 4: Apply ADASYN ONLY to training data to handle class imbalance
adasyn = ADASYN(random_state=42)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train, y_train)

# Test data stays untouched for fair evaluation
X_train = X_train_resampled
y_train = y_train_resampled

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type and experiment info
        mlflow.set_tag("mlflow.runName", f"{model_name}_ADASYN_TFIDF_Bigrams_NoLeakage")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.set_tag("data_handling", "no_leakage_fixed")
        mlflow.set_tag("model_type", "LGBMClassifier")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("vectorizer_type", "TfidfVectorizer")
        mlflow.log_param("ngram_range", str(ngram_range))
        mlflow.log_param("vectorizer_max_features", max_features)
        mlflow.log_param("imbalance_handling", "ADASYN_on_train_only")
        mlflow.log_param("n_estimators", model.n_estimators)
        mlflow.log_param("learning_rate", model.learning_rate)
        mlflow.log_param("max_depth", model.max_depth)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report metrics
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.lightgbm.log_model(model, f"{model_name}_model")
        
        print(f"Run logged successfully for {model_name}")


# Step 5: Optuna objective function for LightGBM
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = LGBMClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


# Step 6: Run Optuna for LightGBM, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=13)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42, verbose=-1)

    # Log the best model with MLflow
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test)
    
    print(f"\nBest Optuna parameters: {best_params}")
    print(f"Best trial accuracy: {study.best_value:.4f}")

# Run the experiment for LightGBM
run_optuna_experiment()

[I 2026-07-21 02:35:41,118] A new study created in memory with name: no-name-e7a33c85-8df3-4711-b5b9-19fe394bb3a1
C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-21 02:36:40,506] Trial 0 finished with value: 0.4515205236601664 and parameters: {'n_estimators': 111, 'learning_rate': 0.0006116587106069266, 'max_depth': 4}. Best is trial 0 with value: 0.4515205236601664.
C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-21 02:37:24,842] Trial 1 finished with value: 0.4937951725078413 and parameters: {'n_estimators': 86, 'learning_rate': 0.000717544711404514, 'max_depth': 8}. Best is trial 1 with value: 0.4937951725078413.
C:\Users\Dell\AppData\Roamin

Run logged successfully for LightGBM
🏃 View run LightGBM_ADASYN_TFIDF_Bigrams_NoLeakage at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/9/runs/363e38497489413cb9525bfc8c62102c
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/9

Best Optuna parameters: {'n_estimators': 181, 'learning_rate': 0.09761679153628776, 'max_depth': 10}
Best trial accuracy: 0.8200
